# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library in Python.

### Dataset Source
The dataset is defined and described by a [Croissant schema](https://mlcommons.org/croissant/) at the URL below. All entities (record sets, fields, columns) are referenced by their unique `@id` to ensure clarity and reproducibility.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR² dataset metadata and all available records using `mlcroissant`. The metadata includes high-level dataset description and the record sets (tables or logical groups) available.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Use the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access and display dataset metadata
metadata = dataset.metadata  # This is a mlcroissant.Metadata object
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets (`@id`), their field `@id`s, and provide a summary of what's available in the dataset.

**Note**: Every entity will be referenced explicitly by its `@id`, as per Croissant best practices and for downstream reproducibility.

In [ ]:
# List all record sets in the dataset, displaying their @id and their fields (by @id)
record_sets = dataset.record_sets  # This is a dictionary mapping @id -> RecordSet
print(f"Available record sets: {list(record_sets.keys())}\n")

for rs_id, rs in record_sets.items():
    print(f"RecordSet @id: {rs_id}")
    print(f"  name: {rs.name}")
    print(f"  fields:")
    for field in rs.fields:
        print(f"    - field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
Extract data from a chosen record set (using its `@id`) into a pandas DataFrame. You can select any available record set for exploration—here, we extract all of them and reference by `@id`.

In [ ]:
dataframes = {}
# We'll extract all record sets (by @id).
for record_set_id in record_sets:
    # records() yields dictionaries (with key: field @id, value: data)
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"---\nLoaded {len(df)} rows from RecordSet @id: {record_set_id}")
    if not df.empty:
        print(f"Columns (field @ids):\n  {df.columns.to_list()}")

# For demonstration, pick the first record set (if exists) for further analysis
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nMain RecordSet @id for downstream analysis: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform data processing and simple EDA. We'll select appropriate numeric fields and group/categorize using available fields.

You'll need to update `numeric_field_id` and `group_field_id` below based on the field `@id`s observed above.

In [ ]:
# Specify the RecordSet @id to use for EDA (edit if needed)
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# As an example, let's try to get the first numeric field @id for analysis
numeric_field_id = None
group_field_id = None

if not df.empty:
    # Try to infer numeric fields from types (if available) or by simple dtype check
    for f in record_sets[record_set_id].fields:
        # Try to identify numeric field from Croissant field or from dataframe dtypes
        f_id = f.id
        # Use the Croissant data_type if possible
        if str(getattr(f, 'data_type', '')).lower() in ('number', 'integer', 'float'):
            numeric_field_id = f_id
            break
    if numeric_field_id is None:
        # Fallback: try columns with numeric dtype
        for c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field_id = c
                break

    # Use the first non-numeric field as grouping variable, if available
    for f in record_sets[record_set_id].fields:
        f_id = f.id
        if f_id != numeric_field_id:
            group_field_id = f_id
            break
else:
    print("The selected RecordSet has no data rows.")

# EDA processing
if numeric_field_id is not None and numeric_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    print(f"Numeric field selected: {numeric_field_id}")
    threshold = df[numeric_field_id].mean()  # Example: use mean as threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} rows\n")
    display(filtered_df.head())

    # Add normalized version of the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("Cannot perform numeric EDA: no numeric field @id found in available data.")

## 5. Visualization
Visualize distributions and relationships between fields in the selected record set. We'll display a histogram of the selected numeric field and, if applicable, a bar plot by group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Histogram of the numeric field
if numeric_field_id is not None and numeric_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Barplot of mean(numeric_field) by group_field, if applicable
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.barplot(
            x=group_field_id,
            y=numeric_field_id,
            data=df,
            estimator='mean',
            ci=None
        )
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
We loaded the FAIR² dataset using `mlcroissant`, inspected its metadata, and explored its available record sets and fields by their `@id`. We extracted data from the record sets, performed simple numeric transformations, grouped by available fields, and visualized distributions. The use of `@id` fields ensures reproducibility and machine-actionable referencing throughout the data science workflow.

More advanced analyses can be performed based on research questions or policy needs—refer to the Croissant schema and the field descriptions within the dataset for deeper insights.